### Here we are going to normalize the view count and add it to the results of the heuristics so we will be able to measure from Player's F side how good a question is without considering the time it was pulished 

In [1]:
import pandas as pd

In [2]:
SUBJECTS = ['united']
LLMS = [
"EleutherAI-pythia-6.9b",
"meta-llama-Llama-3.1-8B",
"meta-llama-Meta-Llama-3-8B-Instruct"
]
ALGORITHMS = ['max_product', 'random_choice', 'nash_bargaining', 'fair_round_robin']
samples = range(1, 51)
weeks=range(1,54)


In [4]:
for subject in SUBJECTS:
    for llm in LLMS:
        for sample in samples:
            for week in weeks:
                file_path = f'bootstrap_dataset/bootstrap_dataset_{subject}_{llm}_sample_{sample}_week_{week}.parquet'
                df=pd.read_parquet(file_path)[['QuestionId','ViewCount']]
                sum_view_count = df['ViewCount'].sum()
                df['NormalizedViewCount'] = df['ViewCount'] / sum_view_count
                for heuristic in ALGORITHMS:
                    df_heuristic=pd.read_parquet(f'selected_questions/selected_{llm}_sample_{sample}_week_{week}_{heuristic}.parquet')
                    df_heuristic = df_heuristic.merge(df[['QuestionId', 'NormalizedViewCount']], on='QuestionId', how='left')
                    df_heuristic.to_parquet(f'selected_questions/selected_{llm}_sample_{sample}_week_{week}_{heuristic}.parquet', index=False)